## Low confidence tokens

In [1]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm.auto import tqdm
from datasets import load_dataset

The workflow is:
	1.	tokenize a list of texts with padding=True
	2.	get logits from each model
	3.	shift logits and labels for next-token prediction
	4.	compute token-level cross-entropy with reduction="none"
	5.	mask out padding tokens
	6.	summarize per-example features
	7.	train logistic regression on those features

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset = load_dataset("UBC-SLIME/colx_531_group_project")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.json:   0%|          | 0.00/102M [00:00<?, ?B/s]

dev.json:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [3]:
# Setup
tokenizer = AutoTokenizer.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lm1 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-135m").to(device)
lm2 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-360m").to(device)

lm1.eval()
lm2.eval()

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (rotary_emb): Lla

In [4]:
# Split data into batches
def chunk_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

In [5]:
# Compute token level losses for whole batch
import torch.nn.functional as F

def get_batch_token_losses(model, texts, max_length=512):
    encodings = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to(device)

    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]

    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    shift_mask = attention_mask[:, 1:]

    token_losses = F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        reduction="none"
    ).reshape(shift_labels.size())

    token_losses = token_losses.detach().float().cpu()
    shift_mask = shift_mask.detach().cpu()

    batch_losses = []
    for i in range(token_losses.size(0)):
        valid_losses = token_losses[i][shift_mask[i] == 1].numpy()
        batch_losses.append(valid_losses)

    return batch_losses

In [6]:
# Build low confidence token features
def summarize_token_losses(token_losses, low_conf_frac=0.2):
    token_losses = np.array(token_losses)

    if len(token_losses) == 0:
        return {
            "mean_loss": np.nan,
            "max_loss": np.nan,
            "p90_loss": np.nan,
            "low_conf_mean_loss": np.nan,
            "low_conf_median_loss": np.nan,
            "n_tokens": 0
        }

    k = max(1, int(len(token_losses) * low_conf_frac))
    worst_losses = np.sort(token_losses)[-k:]   # highest loss = lowest confidence

    return {
        "mean_loss": float(np.mean(token_losses)),
        "max_loss": float(np.max(token_losses)),
        "p90_loss": float(np.percentile(token_losses, 90)),
        "low_conf_mean_loss": float(np.mean(worst_losses)),
        "low_conf_median_loss": float(np.median(worst_losses)),
        "n_tokens": int(len(token_losses))
    }

In [7]:
# Create features from both models and their differences
def extract_batch_features(texts, low_conf_frac=0.2, max_length=512):
    losses1_batch = get_batch_token_losses(lm1, texts, max_length=max_length)
    losses2_batch = get_batch_token_losses(lm2, texts, max_length=max_length)

    batch_features = []

    for losses1, losses2 in zip(losses1_batch, losses2_batch):
        s1 = summarize_token_losses(losses1, low_conf_frac=low_conf_frac)
        s2 = summarize_token_losses(losses2, low_conf_frac=low_conf_frac)

        feats = {
            "lm1_mean_loss": s1["mean_loss"],
            "lm1_max_loss": s1["max_loss"],
            "lm1_p90_loss": s1["p90_loss"],
            "lm1_low_conf_mean_loss": s1["low_conf_mean_loss"],
            "lm1_low_conf_median_loss": s1["low_conf_median_loss"],

            "lm2_mean_loss": s2["mean_loss"],
            "lm2_max_loss": s2["max_loss"],
            "lm2_p90_loss": s2["p90_loss"],
            "lm2_low_conf_mean_loss": s2["low_conf_mean_loss"],
            "lm2_low_conf_median_loss": s2["low_conf_median_loss"],

            "diff_mean_loss": s1["mean_loss"] - s2["mean_loss"],
            "diff_max_loss": s1["max_loss"] - s2["max_loss"],
            "diff_p90_loss": s1["p90_loss"] - s2["p90_loss"],
            "diff_low_conf_mean_loss": s1["low_conf_mean_loss"] - s2["low_conf_mean_loss"],
            "diff_low_conf_median_loss": s1["low_conf_median_loss"] - s2["low_conf_median_loss"],

            "n_tokens": s1["n_tokens"]
        }

        batch_features.append(feats)

    return batch_features

In [8]:
# Build feature dataframes in batches
def build_feature_dataframe_batched(hf_dataset, split_name, batch_size=8, low_conf_frac=0.2, max_length=512, max_examples=None):
    rows = []

    n = len(hf_dataset) if max_examples is None else min(len(hf_dataset), max_examples)

    for start in tqdm(range(0, n, batch_size), desc=f"Extracting {split_name}"):
        batch = hf_dataset[start:start + batch_size]

        texts = batch["text"]
        ids = batch["id"]
        labels = batch["is_member"] if "is_member" in batch else None

        batch_features = extract_batch_features(
            texts,
            low_conf_frac=low_conf_frac,
            max_length=max_length
        )

        for i, feats in enumerate(batch_features):
            feats["id"] = ids[i]
            if labels is not None:
                feats["is_member"] = labels[i]
            rows.append(feats)

    return pd.DataFrame(rows)

In [9]:
# Small example to validate working condition
train_df_small = build_feature_dataframe_batched(
    dataset["train"],
    split_name="train_small",
    batch_size=8,
    low_conf_frac=0.2,
    max_length=512,
    max_examples=1000
)

val_df_small = build_feature_dataframe_batched(
    dataset["validation"],
    split_name="val_small",
    batch_size=8,
    low_conf_frac=0.2,
    max_length=512,
    max_examples=500
)

Extracting train_small:   0%|          | 0/125 [00:00<?, ?it/s]

Extracting val_small:   0%|          | 0/63 [00:00<?, ?it/s]

In [10]:
# Save
train_df_small.to_csv("train_features_small.csv", index=False)
val_df_small.to_csv("val_features_small.csv", index=False)

In [11]:
# Fit the attack model, simple logistic regression
feature_cols = [
    "lm1_mean_loss", "lm1_max_loss", "lm1_p90_loss",
    "lm1_low_conf_mean_loss", "lm1_low_conf_median_loss",
    "lm2_mean_loss", "lm2_max_loss", "lm2_p90_loss",
    "lm2_low_conf_mean_loss", "lm2_low_conf_median_loss",
    "diff_mean_loss", "diff_max_loss", "diff_p90_loss",
    "diff_low_conf_mean_loss", "diff_low_conf_median_loss",
    "n_tokens"
]

X_train = train_df_small[feature_cols].fillna(0)
y_train = train_df_small["is_member"]

X_val = val_df_small[feature_cols].fillna(0)
y_val = val_df_small["is_member"]

attack_model = LogisticRegression(max_iter=1000)
attack_model.fit(X_train, y_train)

val_probs = attack_model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_val, val_preds))
print("F1:", f1_score(y_val, val_preds))
print("ROC AUC:", roc_auc_score(y_val, val_probs))

Accuracy: 0.7400793650793651
F1: 0.7416173570019724
ROC AUC: 0.8202834645669291


In [12]:
# Threshold tuning
best_threshold = 0.5
best_f1 = -1

for t in np.linspace(0.05, 0.95, 91):
    preds = (val_probs >= t).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

Best threshold: 0.3499999999999999
Best F1: 0.7684563758389261


In [13]:
# Only run after checkign it works
# If small version works, actually run in all splits
train_df = build_feature_dataframe_batched(
    dataset["train"],
    split_name="train",
    batch_size=8,
    low_conf_frac=0.2,
    max_length=128
)

val_df = build_feature_dataframe_batched(
    dataset["validation"],
    split_name="validation",
    batch_size=8,
    low_conf_frac=0.2,
    max_length=128
)

test_df = build_feature_dataframe_batched(
    dataset["test"],
    split_name="test",
    batch_size=8,
    low_conf_frac=0.2,
    max_length=128
)

Extracting train:   0%|          | 0/6250 [00:00<?, ?it/s]

Extracting validation:   0%|          | 0/1250 [00:00<?, ?it/s]

Extracting test:   0%|          | 0/1875 [00:00<?, ?it/s]

In [14]:
train_df.to_csv("train_features.csv", index=False)
val_df.to_csv("val_features.csv", index=False)
test_df.to_csv("test_features.csv", index=False)

In [15]:
# Fit model
X_train = train_df[feature_cols].fillna(0)
y_train = train_df["is_member"]

X_val = val_df[feature_cols].fillna(0)
y_val = val_df["is_member"]

attack_model = LogisticRegression(max_iter=1000)
attack_model.fit(X_train, y_train)

val_probs = attack_model.predict_proba(X_val)[:, 1]

best_threshold = 0.5
best_f1 = -1
for t in np.linspace(0.05, 0.95, 91):
    preds = (val_probs >= t).astype(int)
    score = f1_score(y_val, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

In [16]:
X_test = test_df[feature_cols].fillna(0)
test_probs = attack_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

submission = pd.DataFrame({
    "id": test_df["id"],
    "is_member": test_preds
})

In [17]:
from sklearn.metrics import roc_curve
import numpy as np

fpr, tpr, thresholds = roc_curve(y_val, val_probs)

# Find TPR where FPR <= 0.1
target_fpr = 0.1
valid = fpr <= target_fpr

tpr_at_fpr_01 = tpr[valid][-1]

print("TPR@FPR=0.1:", tpr_at_fpr_01)

TPR@FPR=0.1: 0.2926


In [18]:
submission.to_csv("submission.csv", index=False)
print(submission.head())

       id  is_member
0   32656          1
1  137254          0
2  134884          1
3   33065          1
4   34304          1


In [19]:
!ls

sample_data	   train_features.csv	     val_features_small.csv
submission.csv	   train_features_small.csv
test_features.csv  val_features.csv


In [20]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>